# Bias in Bios Concept-QA Training

Train a text Concept-QA answerer on the sampled Bias-in-Bios biographies.

- Inputs: biography text from the sampled splits.
- Targets: OpenAI/regex concept labels from `labels_wide_test_train_validation.csv`.
- Model: sentence-transformer text embeddings + concept description embeddings, scored by `ConceptAnswererMLP`.
- Evaluation: concept-label prediction on validation and test splits.

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "assets" / "concepts" / "bias_in_bios.csv").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")


repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from claq.models import ConceptAnswererMLP
from claq.training import seed_everything

seed_everything(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder_name = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384

sample_dir = repo_root / "artifacts" / "data" / "bias_in_bios_sample"
concepts_path = repo_root / "assets" / "concepts" / "bias_in_bios.csv"
labels_wide_path = sample_dir / "labels_wide_test_train_validation.csv"
models_dir = repo_root / "artifacts" / "models"
runs_dir = repo_root / "artifacts" / "runs"
models_dir.mkdir(parents=True, exist_ok=True)
runs_dir.mkdir(parents=True, exist_ok=True)

SPLITS = ["train", "validation", "test"]
TEXT_COLUMN = "hard_text"

{
    "repo_root": str(repo_root),
    "sample_dir": str(sample_dir),
    "concepts_path": str(concepts_path),
    "labels_wide_path": str(labels_wide_path),
    "device": str(device),
    "encoder_name": encoder_name,
}

In [ ]:
concepts_df = pd.read_csv(concepts_path)
concept_names = concepts_df["concept"].tolist()
concept_texts = concepts_df["description"].tolist()

split_frames = {
    split: pd.read_csv(sample_dir / f"{split}.csv")
    for split in SPLITS
}
labels_wide = pd.read_csv(labels_wide_path)

required_label_columns = {"split", "sample_row", "source_index", "profession_name", "gender_name", *concept_names}
missing_label_columns = sorted(required_label_columns - set(labels_wide.columns))
if missing_label_columns:
    raise ValueError(f"labels_wide is missing columns: {missing_label_columns}")

split_counts = labels_wide["split"].value_counts().reindex(SPLITS, fill_value=0).to_dict()
expected_split_counts = {split: len(frame) for split, frame in split_frames.items()}
if split_counts != expected_split_counts:
    raise ValueError(f"Split counts mismatch. Expected={expected_split_counts}, actual={split_counts}")

{
    "num_concepts": len(concept_names),
    "split_counts": split_counts,
    "concept_kinds": concepts_df["kind"].value_counts().to_dict(),
}

In [ ]:
def build_split_frame(split):
    samples = split_frames[split].copy().reset_index(names="sample_row")
    labels = labels_wide.loc[labels_wide["split"].eq(split)].copy()
    merged = samples.merge(
        labels[["sample_row", "source_index", *concept_names]],
        on=["sample_row", "source_index"],
        how="inner",
        validate="one_to_one",
    )
    if len(merged) != len(samples):
        raise ValueError(f"Merged {len(merged)} rows for {split}, expected {len(samples)}")
    return merged


train_df = build_split_frame("train")
validation_df = build_split_frame("validation")
test_df = build_split_frame("test")
split_dataframes = {"train": train_df, "validation": validation_df, "test": test_df}

{
    split: {
        "rows": len(frame),
        "num_professions": frame["profession_name"].nunique(),
        "positive_labels": int(frame[concept_names].sum().sum()),
    }
    for split, frame in split_dataframes.items()
}

In [ ]:
encoder = SentenceTransformer(encoder_name, device=str(device))

concept_embeddings = encoder.encode(
    concept_texts,
    batch_size=64,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).to(device)

encoded_splits = {}
for split, frame in split_dataframes.items():
    embeddings = encoder.encode(
        frame[TEXT_COLUMN].tolist(),
        batch_size=128,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=True,
    )
    targets = torch.tensor(frame[concept_names].to_numpy(dtype=np.float32))
    profession_ids = torch.tensor(frame["profession"].to_numpy(dtype=np.int64))
    encoded_splits[split] = {
        "embeddings": embeddings.cpu(),
        "targets": targets,
        "profession_ids": profession_ids,
        "profession_names": frame["profession_name"].tolist(),
        "gender_names": frame["gender_name"].tolist(),
        "sample_rows": frame["sample_row"].tolist(),
    }

{
    split: {
        "embeddings": tuple(data["embeddings"].shape),
        "targets": tuple(data["targets"].shape),
    }
    for split, data in encoded_splits.items()
}

In [ ]:
batch_size = 256


def make_dataset(split):
    return TensorDataset(
        encoded_splits[split]["embeddings"].float(),
        encoded_splits[split]["targets"].float(),
    )


train_loader = DataLoader(make_dataset("train"), batch_size=batch_size, shuffle=True)
validation_loader = DataLoader(make_dataset("validation"), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(make_dataset("test"), batch_size=batch_size, shuffle=False)

{
    "train_batches": len(train_loader),
    "validation_batches": len(validation_loader),
    "test_batches": len(test_loader),
    "batch_size": batch_size,
}

In [ ]:
def build_text_concept_qa_inputs(text_embeddings, concept_embeddings):
    batch_size = text_embeddings.size(0)
    num_concepts = concept_embeddings.size(0)
    text_rep = text_embeddings.unsqueeze(1).repeat(1, num_concepts, 1)
    concept_rep = concept_embeddings.unsqueeze(0).repeat(batch_size, 1, 1)
    return torch.cat([text_rep, concept_rep], dim=2).reshape(batch_size * num_concepts, -1)


@torch.no_grad()
def evaluate_concept_qa(model, loader, threshold=0.5):
    model.eval()
    losses = []
    pred_batches = []
    target_batches = []

    for text_batch, target_batch in tqdm(loader, desc="Evaluate", leave=False):
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view_as(target_batch)
        probs = torch.sigmoid(logits)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        losses.append(float(loss.item()))
        pred_batches.append((probs >= threshold).detach().cpu())
        target_batches.append(target_batch.detach().cpu().bool())

    preds = torch.cat(pred_batches)
    targets = torch.cat(target_batches)
    tp = (preds & targets).sum().item()
    fp = (preds & ~targets).sum().item()
    fn = (~preds & targets).sum().item()
    tn = (~preds & ~targets).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)

    return {
        "loss": float(np.mean(losses)),
        "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_rate_pred": float(preds.float().mean().item()),
        "positive_rate_target": float(targets.float().mean().item()),
    }

In [ ]:
model_config = {
    "embed_dim": embedding_dim,
    "hidden_dims": (256, 128, 64),
    "dropout": 0.1,
    "use_batch_norm": True,
}
concept_qa_model = ConceptAnswererMLP(**model_config).to(device)
optimizer = torch.optim.AdamW(concept_qa_model.parameters(), lr=1e-3, weight_decay=1e-4)
num_epochs = 50

concept_qa_history = []
best_validation_f1 = -1.0
best_epoch = None
best_state_dict = None

for epoch in tqdm(range(1, num_epochs + 1), desc="Concept-QA epochs"):
    concept_qa_model.train()
    train_losses = []

    for text_batch, target_batch in tqdm(train_loader, desc=f"Train epoch {epoch}", leave=False):
        text_batch = text_batch.to(device)
        target_batch = target_batch.to(device)

        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = concept_qa_model(inputs).view_as(target_batch)
        loss = F.binary_cross_entropy_with_logits(logits, target_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_losses.append(float(loss.item()))

    validation_metrics = evaluate_concept_qa(concept_qa_model, validation_loader)
    epoch_metrics = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **{f"validation_{key}": value for key, value in validation_metrics.items()},
    }
    concept_qa_history.append(epoch_metrics)

    if validation_metrics["f1"] > best_validation_f1:
        best_validation_f1 = validation_metrics["f1"]
        best_epoch = epoch
        best_state_dict = {
            name: parameter.detach().cpu().clone()
            for name, parameter in concept_qa_model.state_dict().items()
        }

if best_state_dict is None:
    raise RuntimeError("No best model state was recorded.")

concept_qa_model.load_state_dict(best_state_dict)
history_df = pd.DataFrame(concept_qa_history)
test_metrics = evaluate_concept_qa(concept_qa_model, test_loader)

best_summary = {
    "best_epoch": best_epoch,
    "best_validation_f1": best_validation_f1,
    "test_metrics_at_best_validation_epoch": test_metrics,
}

history_df.tail(10), best_summary

In [ ]:
@torch.no_grad()
def collect_concept_predictions(model, loader):
    model.eval()
    prob_batches = []
    target_batches = []

    for text_batch, target_batch in tqdm(loader, desc="Collect predictions", leave=False):
        text_batch = text_batch.to(device)
        inputs = build_text_concept_qa_inputs(text_batch, concept_embeddings)
        logits = model(inputs).view(target_batch.size(0), len(concept_names))
        prob_batches.append(torch.sigmoid(logits).cpu())
        target_batches.append(target_batch.cpu().bool())

    return torch.cat(prob_batches), torch.cat(target_batches)


def binary_metrics_from_predictions(probs, targets, threshold):
    preds = probs >= threshold
    tp = (preds & targets).sum().item()
    fp = (preds & ~targets).sum().item()
    fn = (~preds & targets).sum().item()
    tn = (~preds & ~targets).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {
        "threshold": float(threshold),
        "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "positive_rate_pred": float(preds.float().mean().item()),
        "positive_rate_target": float(targets.float().mean().item()),
    }


validation_probs, validation_targets = collect_concept_predictions(concept_qa_model, validation_loader)
test_probs, test_targets = collect_concept_predictions(concept_qa_model, test_loader)

threshold_grid = np.round(np.arange(0.05, 0.951, 0.01), 2)
threshold_search = pd.DataFrame(
    [binary_metrics_from_predictions(validation_probs, validation_targets, threshold) for threshold in threshold_grid]
)
best_threshold_row = threshold_search.sort_values(["f1", "precision"], ascending=False).iloc[0]
best_global_threshold = float(best_threshold_row["threshold"])

test_metrics_at_half_threshold = binary_metrics_from_predictions(test_probs, test_targets, threshold=0.5)
tuned_test_metrics = binary_metrics_from_predictions(test_probs, test_targets, threshold=best_global_threshold)

per_concept_rows = []
for idx, concept in enumerate(concept_names):
    concept_probs = test_probs[:, idx]
    concept_targets = test_targets[:, idx]
    metrics = binary_metrics_from_predictions(concept_probs, concept_targets, threshold=best_global_threshold)
    per_concept_rows.append(
        {
            "concept": concept,
            "kind": concepts_df.loc[idx, "kind"],
            "target_positive_count": int(concept_targets.sum().item()),
            "pred_positive_count": int((concept_probs >= best_global_threshold).sum().item()),
            **metrics,
        }
    )

per_concept_diagnostics = pd.DataFrame(per_concept_rows).sort_values(
    ["f1", "target_positive_count"], ascending=[True, True]
)

{
    "best_global_threshold": best_global_threshold,
    "validation_metrics_at_best_threshold": best_threshold_row.to_dict(),
    "test_metrics_at_half_threshold": test_metrics_at_half_threshold,
    "tuned_test_metrics": tuned_test_metrics,
    "weakest_test_concepts": per_concept_diagnostics.head(10).to_dict(orient="records"),
}

In [ ]:
@torch.no_grad()
def predict_concepts(split="test", row_index=0):
    concept_qa_model.eval()
    text_embedding = encoded_splits[split]["embeddings"][row_index : row_index + 1].to(device)
    target = encoded_splits[split]["targets"][row_index].numpy()
    inputs = build_text_concept_qa_inputs(text_embedding, concept_embeddings)
    scores = torch.sigmoid(concept_qa_model(inputs).view(1, len(concept_names))).squeeze(0).cpu().numpy()
    order = np.argsort(scores)[::-1]

    return pd.DataFrame(
        {
            "concept": [concept_names[idx] for idx in order],
            "kind": [concepts_df.loc[idx, "kind"] for idx in order],
            "score": scores[order],
            "target": target[order].astype(int),
        }
    )


review_split = "test"
review_row_index = 0
print(split_dataframes[review_split].iloc[review_row_index][TEXT_COLUMN])
predict_concepts(review_split, review_row_index).head(12)

In [ ]:
experiment_name = "bias_in_bios_concept_answerer_mlp_openai_labels"
checkpoint_path = models_dir / f"concept_qa_{experiment_name}.pt"
history_path = runs_dir / f"concept_qa_{experiment_name}_history.json"
diagnostics_path = runs_dir / f"concept_qa_{experiment_name}_per_concept_diagnostics.csv"

selected_threshold = globals().get("best_global_threshold", 0.5)
selected_test_metrics = globals().get("tuned_test_metrics", test_metrics)

checkpoint_payload = {
    "model_state_dict": concept_qa_model.state_dict(),
    "model_class": "ConceptAnswererMLP",
    "model_config": model_config,
    "embedding_dim": embedding_dim,
    "encoder_name": encoder_name,
    "concepts": concepts_df.to_dict(orient="records"),
    "concepts_path": str(concepts_path.relative_to(repo_root)),
    "labels_wide_path": str(labels_wide_path.relative_to(repo_root)),
    "concept_embeddings": concept_embeddings.detach().cpu(),
    "selection_metric": "validation_f1",
    "best_epoch": best_epoch,
    "best_validation_f1": best_validation_f1,
    "decision_threshold": selected_threshold,
    "test_metrics": test_metrics,
    "selected_threshold_test_metrics": selected_test_metrics,
}
torch.save(checkpoint_payload, checkpoint_path)

if "per_concept_diagnostics" in globals():
    per_concept_diagnostics.to_csv(diagnostics_path, index=False, lineterminator="\n")

history_payload = {
    "experiment_name": experiment_name,
    "model_class": "ConceptAnswererMLP",
    "model_config": model_config,
    "encoder_name": encoder_name,
    "num_epochs": num_epochs,
    "batch_size": batch_size,
    "selection_metric": "validation_f1",
    "best_epoch": best_epoch,
    "best_validation_f1": best_validation_f1,
    "decision_threshold": selected_threshold,
    "history": concept_qa_history,
    "test_metrics": test_metrics,
    "selected_threshold_test_metrics": selected_test_metrics,
    "split_counts": {split: len(frame) for split, frame in split_dataframes.items()},
    "num_concepts": len(concept_names),
}
with history_path.open("w", encoding="utf-8") as handle:
    json.dump(history_payload, handle, indent=2)

{
    "checkpoint": str(checkpoint_path.relative_to(repo_root)),
    "history": str(history_path.relative_to(repo_root)),
    "per_concept_diagnostics": str(diagnostics_path.relative_to(repo_root)) if "per_concept_diagnostics" in globals() else None,
}